# Stitch viewer: tiles, design, contours and design errors, before and after stitching

Runs the pipeline on the chosen tiles and opens one interactive window (pyqtgraph, OpenGL):

1. **Per tile** (`process_tile`): load the SEM image, detect contacts (Otsu), refine the contours
   to the maximum gradient, read the design contacts from the tile's `.oas` in the tone (normal /
   reversed) that matches the SEM contacts. Tiles where neither tone matches are flagged.
2. **Stitching** (`stitch_tiles`): match the contacts in every overlap, robust B − A translation
   per pair, one translation correction per tile. Failed overlaps and unstitched tiles are listed.
3. **Design errors** (`design_errors`): per tile, stitched SEM centres matched to the design
   centres; raw error SEM − design (no affine removed yet). Tiles without errors are listed.
4. **Viewer** (`StitchViewer`): left, SEM images, design contours/centres, Otsu and refined
   contours/centres, overlap boxes and overlap outliers; right, the error map, sharing the zoom.
   *Placement* switches between the nominal (metadata) and the stitched positions. Layers and
   single tiles can be hidden on the right.

The window blocks this notebook until it is closed. Positions in µm, errors in nm.

In [ ]:
DATA_DIR = r"A:\Stitch_monitor\In_cell_without_PTM_20260914\260910_1143_Norm"
TILES = None              # None = all tiles, or a list of indices in read_tile_index order, e.g. list(range(10))
LAYER = (100, 0)          # (layer, datatype) of the contacts in the .oas
SEARCH_PX = 5.0           # refinement searches this many pixels in/out along the contour normal
DESIGN_MATCH_NM = 25.0    # pair design and SEM contacts closer than this (tone check and design errors)
MIN_MATCH_FRACTION = 0.5  # a design tone is accepted if at least this fraction of SEM contacts match it
STITCH_REFINED = True     # stitch with the refined centres (False: Otsu centres)
MAX_MATCH_NM = 25.0       # pair overlap contacts closer than this: the largest B - A stage offset (must be < pitch / 2)
OUTLIER_FACTOR = 3.0      # overlap outlier if residual > this x the median residual
MIN_MATCHED = 5           # pairs with fewer matched contacts are not used (reported below)
ARROW_SCALE = 100         # error-map arrows are drawn this many times longer than the error
USE_OPENGL = True         # set False if the viewer window stays black

In [ ]:
import time

import numpy as np
import pandas as pd
import pyqtgraph as pg

from affine_ransac.io.design import load_layout, read_polygons
from affine_ransac.io.metadata import read_tile_index
from affine_ransac.pipeline import process_tile, stitch_tiles
from affine_ransac.registration import design_errors, error_summary
from affine_ransac.view_stitch import StitchViewer

## 1. Load the tiles and detect their contacts

In [ ]:
all_tiles = read_tile_index(DATA_DIR)
records = all_tiles if TILES is None else [all_tiles[i] for i in TILES]
tile_ids = [r.tile_id for r in records]

start = time.perf_counter()
tiles = [
    process_tile(r.image_path, r.oas_path, (r.center_x_nm, r.center_y_nm), (r.fov_x_nm, r.fov_y_nm),
                 LAYER, SEARCH_PX, DESIGN_MATCH_NM, MIN_MATCH_FRACTION)
    for r in records
]
print(f"{len(tiles)} of {len(all_tiles)} tiles in {time.perf_counter() - start:.1f} s")

per_tile = pd.DataFrame({
    "tile": tile_ids,
    "design": [len(t.design.centers) for t in tiles],
    "otsu": [len(t.otsu.centers) for t in tiles],
    "refined": [len(t.refined.centers) for t in tiles],
    "tone": ["reversed" if t.design.reversed else "normal" for t in tiles],
    "match_fraction": [t.design.match_fraction for t in tiles],
    "design_ok": [t.design.ok for t in tiles],
})
print(f"Design tone: {(per_tile.tone == 'normal').sum()} normal, {(per_tile.tone == 'reversed').sum()} reversed")
flagged = per_tile[~per_tile.design_ok]
if len(flagged):
    print(f"\n!! {len(flagged)} tile(s) FLAGGED: neither design tone matches the SEM (or no .oas):")
    display(flagged.round(2))
per_tile.describe().round(2)

## 2. Stitching

In [ ]:
centers = np.array([t.center_nm for t in tiles])
fovs = np.array([t.fov_nm for t in tiles])
stitch = stitch_tiles([t.sem_points_nm(STITCH_REFINED) for t in tiles], centers, fovs,
                      MAX_MATCH_NM, OUTLIER_FACTOR, MIN_MATCHED)

failed = [r for r in stitch.rejected if r.failed]
print(f"{len(stitch.pairs)} overlapping pairs used, {len(failed)} not matched, "
      f"{len(stitch.rejected) - len(failed)} skipped (fewer than {MIN_MATCHED} contacts in the overlap)")
print(f"Corrections: |x| max {np.nanmax(np.abs(stitch.corrections[:, 0])):.2f} nm, "
      f"|y| max {np.nanmax(np.abs(stitch.corrections[:, 1])):.2f} nm")
if failed or len(stitch.unplaced):
    print(f"\n!! STITCHING INCOMPLETE. Tiles not stitched (no correction): {[tile_ids[k] for k in stitch.unplaced]}")
    print("!! Overlaps whose contacts did not match:")
    display(pd.DataFrame([{"tile_a": tile_ids[r.i], "tile_b": tile_ids[r.j], "contacts_in_overlap": r.in_box,
                           "matched": r.matched} for r in failed]))

pd.DataFrame([{
    "tile_a": tile_ids[p.i], "tile_b": tile_ids[p.j], "matched": len(p.ia),
    "outliers": int((~p.fit.inliers).sum()), "shift_x_nm": p.fit.shift[0], "shift_y_nm": p.fit.shift[1],
    "rms_nm": p.fit.rms(),
} for p in stitch.pairs]).round(2)

## 3. Design errors (raw: stitched SEM − design, no affine removed)

A contact seen by two tiles in an overlap has one entry per tile. The common offset is partly
arbitrary (stitching corrections average zero), so look at the pattern: a uniform shift, a
rotation (arrows circling) or a scale (arrows pointing outward/inward).

In [ ]:
errors = design_errors([t.sem_points_nm(STITCH_REFINED) for t in tiles], [t.design_points_nm() for t in tiles],
                       stitch.corrections, [t.design.ok for t in tiles], DESIGN_MATCH_NM)

if errors.skipped:
    print(f"!! {len(errors.skipped)} tile(s) WITHOUT design errors:")
    display(pd.DataFrame([{"tile": tile_ids[k], "reason": reason} for k, reason in errors.skipped.items()]))
pd.Series(error_summary(errors.error_nm)).round(2)

## 4. Viewer (closes with the window)

Left: SEM and design. Right: the error map (dot = design position, line = error × `ARROW_SCALE`,
adjustable in the panel). Both plots share zoom and pan.

In [ ]:
design_polygons = [read_polygons(load_layout(r.oas_path), *LAYER) if r.oas_path else [] for r in records]

app = pg.mkQApp("Stitch viewer")
viewer = StitchViewer(tiles, tile_ids, design_polygons, stitch, refined=STITCH_REFINED, use_opengl=USE_OPENGL,
                      errors=errors, arrow_scale=ARROW_SCALE)
viewer.show()
app.exec()